# Assignment 4: Convolutional Neural Network for Tomato Disease Classification

## Aim

To design and implement a Convolutional Neural Network (CNN) using TensorFlow/Keras for classifying tomato leaf images into different disease and healthy categories.

## Problem Statement

Plant diseases can significantly affect agricultural productivity. Manual identification of plant diseases from leaf images can be time-consuming and requires expert knowledge.

In this assignment, a Convolutional Neural Network (CNN) is developed to automatically classify tomato leaf images into different disease categories using the PlantVillage dataset.

The model performs image preprocessing, data augmentation, feature extraction using convolutional layers, classification, and performance evaluation.

## Objectives

- To understand the application of CNNs for image classification.
- To preprocess and normalize tomato leaf images.
- To apply data augmentation to improve model generalization.
- To design and train a CNN using TensorFlow/Keras.
- To evaluate the trained model using accuracy and loss.
- To analyze model performance using a confusion matrix and classification report.
- To test the model on unseen tomato leaf images.

## 1. Dataset Description

The PlantVillage dataset is used for tomato leaf disease classification.

The dataset contains images of plant leaves belonging to different crops and disease categories. For this assignment, only the tomato-related classes are selected.

### Tomato Classes

The model will classify images into the following 10 categories:

1. Tomato Bacterial Spot
2. Tomato Early Blight
3. Tomato Healthy
4. Tomato Late Blight
5. Tomato Leaf Mold
6. Tomato Septoria Leaf Spot
7. Tomato Spider Mites / Two-Spotted Spider Mite
8. Tomato Target Spot
9. Tomato Tomato Mosaic Virus
10. Tomato Tomato Yellow Leaf Curl Virus

The dataset contains separate `train` and `val` directories. The original training data is further divided into training and validation subsets, while the original validation directory is used as an independent test set.

### Dataset Structure

```text
PlantVillage/
├── train/
│   ├── Tomato___Bacterial_spot/
│   ├── Tomato___Early_blight/
│   ├── ...
│   └── Tomato___Tomato_Yellow_Leaf_Curl_Virus/
│
└── val/
    ├── Tomato___Bacterial_spot/
    ├── Tomato___Early_blight/
    ├── ...
    └── Tomato___Tomato_Yellow_Leaf_Curl_Virus/

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

print("TensorFlow Version:", tf.__version__)

## 2. Reproducibility

Random seeds are fixed so that dataset splitting and model experiments produce consistent results as much as possible.

In [ ]:
# Set random seeds for reproducibility

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed set to:", SEED)

## 3. Dataset Path Configuration

Specify the path to the extracted PlantVillage dataset.

The dataset should contain two directories:

- `train`
- `val`

In [ ]:
from pathlib import Path

# Dataset root directory
DATASET_ROOT = Path("PlantVillage")

# Train and validation directories
TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "val"

print("Dataset root:", DATASET_ROOT.resolve())
print("Training directory:", TRAIN_DIR.resolve())
print("Validation directory:", VAL_DIR.resolve())

# Check that directories exist
if not TRAIN_DIR.exists():
    raise FileNotFoundError(
        f"Training directory not found: {TRAIN_DIR.resolve()}"
    )

if not VAL_DIR.exists():
    raise FileNotFoundError(
        f"Validation directory not found: {VAL_DIR.resolve()}"
    )

print("\nDataset directories found successfully!")

## 4. Selecting Tomato Classes

The PlantVillage dataset contains images belonging to multiple crops and disease categories.

For this assignment, only the tomato-related classes are selected. The notebook identifies these classes automatically by checking for directory names beginning with `Tomato___`.

The original dataset is not modified; only the required tomato classes are selected programmatically.

In [ ]:
# Find all Tomato class directories in the training dataset

tomato_train_dirs = sorted([
    folder for folder in TRAIN_DIR.iterdir()
    if folder.is_dir() and folder.name.startswith("Tomato___")
])

# Find all Tomato class directories in the validation/test dataset

tomato_val_dirs = sorted([
    folder for folder in VAL_DIR.iterdir()
    if folder.is_dir() and folder.name.startswith("Tomato___")
])

# Extract class names
class_names = sorted([
    folder.name for folder in tomato_train_dirs
])

print("Number of Tomato classes:", len(class_names))

print("\nTomato Classes:")
print("-" * 50)

for i, class_name in enumerate(class_names):
    print(f"{i}: {class_name}")

print("\nClasses found in training directory:", len(tomato_train_dirs))
print("Classes found in validation directory:", len(tomato_val_dirs))

## 5. Class Name Mapping

The original directory names are machine-friendly names such as `Tomato___Early_blight`.

For analysis and visualization, these names are converted into readable class labels by removing the `Tomato___` prefix and replacing underscores with spaces.

In [ ]:
# Create readable class names

def clean_class_name(name):
    name = name.replace("Tomato___", "")
    name = name.replace("_", " ")
    return name


readable_class_names = [
    clean_class_name(name)
    for name in class_names
]

# Create mappings between class names and numerical labels

class_to_index = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

index_to_class = {
    index: readable_class_names[index]
    for index in range(len(class_names))
}

print("Class Mapping:")
print("-" * 50)

for index, class_name in enumerate(readable_class_names):
    print(f"{index}: {class_name}")

## 6. Collecting Image Paths

The paths of all tomato leaf images are collected along with their corresponding class labels.

Two datasets are created:

- **Training data:** Images from the `train` directory.
- **Test data:** Images from the `val` directory.

The training data will later be divided into training and validation subsets.

In [ ]:
# Supported image file extensions

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".gif"
}


def collect_images(directory, class_names):
    records = []

    for class_name in class_names:

        class_dir = directory / class_name

        if not class_dir.exists():
            print(f"Warning: {class_name} not found in {directory}")
            continue

        # Recursively find image files
        for image_path in class_dir.rglob("*"):

            if (
                image_path.is_file()
                and image_path.suffix.lower() in IMAGE_EXTENSIONS
            ):
                records.append({
                    "filepath": str(image_path),
                    "class_name": class_name,
                    "label": class_to_index[class_name]
                })

    return pd.DataFrame(records)


# Collect training images
train_full_df = collect_images(
    TRAIN_DIR,
    class_names
)

# Collect test images from the original val directory
test_df = collect_images(
    VAL_DIR,
    class_names
)


print("Total training images:", len(train_full_df))
print("Total test images:", len(test_df))

print("\nTraining images by class:")
print(
    train_full_df["class_name"]
    .value_counts()
    .sort_index()
)

print("\nTest images by class:")
print(
    test_df["class_name"]
    .value_counts()
    .sort_index()
)

## 7. Creating Training and Validation Sets

The images from the original `train` directory are divided into two subsets:

- **85% Training**
- **15% Validation**

A stratified split is used so that each class maintains approximately the same proportion in both subsets.

The original `val` directory is kept completely separate and is used as the final test dataset. Therefore, the test images are not used during model training or validation.

In [ ]:
# Create a stratified training-validation split

train_df, validation_df = train_test_split(
    train_full_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_full_df["label"]
)

# Reset DataFrame indices
train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Display dataset sizes
print("Dataset Split")
print("-" * 40)

print(f"Training   : {len(train_df):,} images")
print(f"Validation : {len(validation_df):,} images")
print(f"Testing    : {len(test_df):,} images")

print("\nTotal images:", len(train_df) + len(validation_df) + len(test_df))

print("\nTraining percentage:",
      f"{len(train_df) / len(train_full_df) * 100:.2f}%")

print("Validation percentage:",
      f"{len(validation_df) / len(train_full_df) * 100:.2f}%")

print("Test percentage:",
      f"{len(test_df) / (len(train_full_df) + len(test_df)) * 100:.2f}%")

## 8. Class Distribution

The number of training images in each tomato disease category is analyzed to understand the distribution of the dataset.

This analysis is important because the dataset is not perfectly balanced. Some disease categories contain significantly more images than others.

The class distribution will also be considered during model training using class weights.

In [ ]:
# Calculate the number of training images in each class

class_distribution = (
    train_df["class_name"]
    .value_counts()
    .reindex(class_names)
)

# Create a readable DataFrame

distribution_df = pd.DataFrame({
    "Class": readable_class_names,
    "Number of Training Images": class_distribution.values
})

distribution_df

In [ ]:
# Plot the training class distribution

plt.figure(figsize=(12, 6))

plt.bar(
    readable_class_names,
    class_distribution.values
)

plt.title("Training Dataset - Tomato Disease Class Distribution")
plt.xlabel("Disease Class")
plt.ylabel("Number of Training Images")

plt.xticks(
    rotation=60,
    ha="right"
)

plt.tight_layout()
plt.show()

## 9. Visualizing Sample Images

Sample images from each tomato disease category are displayed to understand the visual characteristics of the dataset.

One representative image from each of the 10 classes is selected and displayed together with its corresponding class label.

In [ ]:
# Display one sample image from each Tomato disease class

fig, axes = plt.subplots(
    2,
    5,
    figsize=(18, 8)
)

axes = axes.flatten()

for index, class_name in enumerate(class_names):

    # Get images belonging to the current class
    class_images = train_full_df[
        train_full_df["class_name"] == class_name
    ]

    # Select the first image
    image_path = class_images.iloc[0]["filepath"]

    # Read image
    image = plt.imread(image_path)

    # Display image
    axes[index].imshow(image)
    axes[index].set_title(
        readable_class_names[index],
        fontsize=10
    )
    axes[index].axis("off")

plt.suptitle(
    "Sample Images from Tomato Disease Classes",
    fontsize=16
)

plt.tight_layout()
plt.show()

## 10. Image Preprocessing

CNN models require input images to have a consistent size and numerical format.

The following preprocessing steps will be applied:

- Resize every image to **128 × 128 pixels**.
- Use **3 color channels (RGB)**.
- Normalize pixel values from the range **0–255** to **0–1**.
- Load images in batches to improve training efficiency.

A batch size of 32 is used for training.

In [ ]:
# Image preprocessing configuration

IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3

BATCH_SIZE = 32

print("Image Height :", IMG_HEIGHT)
print("Image Width  :", IMG_WIDTH)
print("Channels     :", CHANNELS)
print("Batch Size   :", BATCH_SIZE)

## 11. Creating TensorFlow Data Pipelines

TensorFlow `tf.data` pipelines are used to efficiently load, preprocess, batch, and prefetch the images.

Each image is:

1. Loaded from its file path.
2. Decoded as an RGB image.
3. Resized to 128 × 128 pixels.
4. Converted to floating-point values.
5. Normalized from the range 0–255 to 0–1.

The training dataset is shuffled, while validation and test datasets are kept in a fixed order.

In [ ]:
# Function to load and preprocess a single image

def load_image(image_path, label):
    # Read image file
    image = tf.io.read_file(image_path)

    # Decode image as RGB
    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    # Resize image
    image = tf.image.resize(
        image,
        [IMG_HEIGHT, IMG_WIDTH]
    )

    # Normalize pixel values from 0-255 to 0-1
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


# Function to create a TensorFlow dataset

def create_dataset(dataframe, shuffle=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label"].values

    # Create TensorFlow dataset
    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    # Load and preprocess images
    dataset = dataset.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Shuffle only the training dataset
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=SEED
        )

    # Create batches
    dataset = dataset.batch(BATCH_SIZE)

    # Prefetch batches for better performance
    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


# Create training dataset
train_dataset = create_dataset(
    train_df,
    shuffle=True
)

# Create validation dataset
validation_dataset = create_dataset(
    validation_df,
    shuffle=False
)

# Create test dataset
test_dataset = create_dataset(
    test_df,
    shuffle=False
)

print("TensorFlow data pipelines created successfully.")

## 12. Verifying the Dataset Pipeline

Before proceeding to model training, one batch of images and labels is inspected.

This verifies that:

- Images are loaded correctly.
- Images have the expected shape of 128 × 128 × 3.
- Labels are loaded correctly.
- Pixel values have been normalized to the range 0–1.

In [ ]:
# Get one batch from the training dataset

images, labels = next(iter(train_dataset))

print("Image batch shape :", images.shape)
print("Label batch shape :", labels.shape)

print("\nPixel value range:")
print("Minimum pixel value:", tf.reduce_min(images).numpy())
print("Maximum pixel value:", tf.reduce_max(images).numpy())

print("\nLabel range:")
print("Minimum label:", tf.reduce_min(labels).numpy())
print("Maximum label:", tf.reduce_max(labels).numpy())

## 13. Data Augmentation

Data augmentation is applied to the training images to improve the model's ability to generalize to new images.

The following transformations are used:

- Random horizontal flipping
- Random rotation
- Random zoom
- Random contrast adjustment

These transformations are applied dynamically during training. The validation and test images are not augmented, ensuring that model evaluation is performed on the original images.

In [ ]:
# Create the data augmentation pipeline

data_augmentation = tf.keras.Sequential([
    
    # Randomly flip images horizontally
    tf.keras.layers.RandomFlip(
        mode="horizontal"
    ),
    
    # Randomly rotate images
    tf.keras.layers.RandomRotation(
        factor=0.10
    ),
    
    # Randomly zoom images
    tf.keras.layers.RandomZoom(
        height_factor=0.10,
        width_factor=0.10
    ),
    
    # Randomly change image contrast
    tf.keras.layers.RandomContrast(
        factor=0.10
    )

], name="data_augmentation")


print("Data augmentation pipeline created successfully.")
print("\nAugmentation layers:")

for layer in data_augmentation.layers:
    print("-", layer.name)

## 14. Visualizing Data Augmentation

To understand the effect of data augmentation, a sample training image is transformed multiple times using the augmentation pipeline.

The generated images demonstrate how random flipping, rotation, zooming, and contrast adjustment can produce different variations of the same leaf image.

This helps increase the diversity of the training data and can improve the model's ability to generalize to unseen images.

In [ ]:
# Visualize multiple augmented versions of a sample image

# Select the first image from the current training batch
sample_image = images[0]

plt.figure(figsize=(12, 8))

for i in range(6):

    # Apply augmentation
    augmented_image = data_augmentation(
        tf.expand_dims(sample_image, axis=0),
        training=True
    )[0]

    # Display augmented image
    plt.subplot(2, 3, i + 1)

    plt.imshow(
        tf.clip_by_value(
            augmented_image,
            0.0,
            1.0
        )
    )

    plt.axis("off")
    plt.title(f"Augmented Image {i + 1}")

plt.suptitle(
    "Data Augmentation Examples",
    fontsize=16
)

plt.tight_layout()
plt.show()

## 15. CNN Architecture

A custom Convolutional Neural Network (CNN) is designed using TensorFlow/Keras for classifying the 10 tomato leaf categories.

The CNN consists of:

- Input layer for 128 × 128 RGB images
- Data augmentation layer
- Three convolutional layers for feature extraction
- ReLU activation functions
- Max pooling layers for dimensionality reduction
- Flatten layer to convert feature maps into a one-dimensional vector
- Dense layer for learning higher-level representations
- Dropout layer to reduce overfitting
- Softmax output layer with 10 neurons for the 10 classes

### Architecture

Input Image
→ Data Augmentation
→ Conv2D (32 filters)
→ MaxPooling
→ Conv2D (64 filters)
→ MaxPooling
→ Conv2D (128 filters)
→ MaxPooling
→ Flatten
→ Dense (128 neurons)
→ Dropout
→ Output (10 classes)

In [ ]:
# Number of output classes
NUM_CLASSES = len(class_names)

# Build the CNN model

model = tf.keras.Sequential([
    
    # Input layer
    tf.keras.layers.Input(
        shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)
    ),

    # Data augmentation
    data_augmentation,

    # Convolutional Block 1
    tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    ),

    tf.keras.layers.MaxPooling2D(
        pool_size=(2, 2)
    ),

    # Convolutional Block 2
    tf.keras.layers.Conv2D(
        filters=64,
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    ),

    tf.keras.layers.MaxPooling2D(
        pool_size=(2, 2)
    ),

    # Convolutional Block 3
    tf.keras.layers.Conv2D(
        filters=128,
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    ),

    tf.keras.layers.MaxPooling2D(
        pool_size=(2, 2)
    ),

    # Convert feature maps into a vector
    tf.keras.layers.Flatten(),

    # Fully connected layer
    tf.keras.layers.Dense(
        units=128,
        activation="relu"
    ),

    # Dropout to reduce overfitting
    tf.keras.layers.Dropout(
        rate=0.5
    ),

    # Output layer
    tf.keras.layers.Dense(
        units=NUM_CLASSES,
        activation="softmax"
    )
])

print("CNN model created successfully.")
print("Number of output classes:", NUM_CLASSES)

## 16. Model Summary

The model summary is displayed to examine the CNN architecture, including:

- Layer types
- Output shapes
- Number of trainable parameters
- Number of non-trainable parameters

The summary helps verify that the CNN architecture has been constructed correctly before training.

In [ ]:
# Display the CNN architecture

model.summary()

## 17. Model Compilation

The CNN is compiled before training.

The following configuration is used:

- **Optimizer:** Adam
- **Learning Rate:** 0.001
- **Loss Function:** Sparse Categorical Crossentropy
- **Evaluation Metric:** Accuracy

The Adam optimizer is selected because it provides adaptive learning rates and generally performs well for image classification tasks.

Sparse categorical crossentropy is used because the target labels are represented as integer class indices from 0 to 9.

In [ ]:
# Compile the CNN model

LEARNING_RATE = 0.001

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    
    loss="sparse_categorical_crossentropy",
    
    metrics=[
        "accuracy"
    ]
)

print("Model compiled successfully.")

print("\nTraining Configuration")
print("-" * 30)
print("Optimizer    : Adam")
print("Learning Rate:", LEARNING_RATE)
print("Loss Function: Sparse Categorical Crossentropy")
print("Metric       : Accuracy")

## 18. Class Weights

The training dataset is imbalanced across the 10 tomato disease classes.

For example, `Tomato Yellow Leaf Curl Virus` has considerably more training images than `Tomato mosaic virus`.

To reduce the effect of this imbalance, class weights are calculated and provided to the model during training.

Classes with fewer images receive higher weights, while classes with more images receive lower weights. This encourages the CNN to pay appropriate attention to all disease categories.

In [ ]:
# Calculate class weights to handle class imbalance

class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
)

total_samples = len(train_df)

class_weights = {}

for class_index in range(NUM_CLASSES):
    
    count = class_counts.get(
        class_index,
        1
    )
    
    class_weights[class_index] = (
        total_samples /
        (NUM_CLASSES * count)
    )


# Display class weights

class_weights_df = pd.DataFrame({
    "Class Index": list(class_weights.keys()),
    "Class": [
        readable_class_names[i]
        for i in class_weights.keys()
    ],
    "Training Images": [
        class_counts.get(i, 0)
        for i in class_weights.keys()
    ],
    "Class Weight": [
        class_weights[i]
        for i in class_weights.keys()
    ]
})

display(
    class_weights_df.round(4)
)

## 19. Model Training

The CNN is trained using the training dataset and evaluated on the validation dataset after each epoch.

Two callbacks are used:

- **EarlyStopping:** Stops training when validation loss does not improve for several consecutive epochs and restores the best model weights.
- **ReduceLROnPlateau:** Reduces the learning rate when validation loss stops improving.

Class weights are supplied during training to reduce the effect of class imbalance.

The maximum number of epochs is set to 30.

In [ ]:
# Training configuration

EPOCHS = 30

# Stop training if validation loss does not improve
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate when validation loss stops improving
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

print("Starting CNN training...")
print(f"Maximum epochs: {EPOCHS}")
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)

# Train the model
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

print("\nTraining completed.")

## 20. Training History

The training history records the performance of the CNN during each epoch.

Two graphs are plotted:

1. **Training and Validation Accuracy**
2. **Training and Validation Loss**

These graphs help analyze the learning behavior of the model and identify signs of underfitting or overfitting.

The best model weights were restored from the epoch with the lowest validation loss.

In [ ]:
# Convert training history into a DataFrame

history_df = pd.DataFrame(history.history)

print("Training history:")
print("-" * 40)

display(
    history_df.round(4)
)

In [ ]:
# Plot training and validation accuracy

plt.figure(figsize=(10, 6))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

# Mark the best validation accuracy
best_epoch = np.argmax(
    history.history["val_accuracy"]
)

best_val_accuracy = history.history["val_accuracy"][best_epoch]

plt.scatter(
    best_epoch,
    best_val_accuracy,
    s=80,
    label=f"Best Validation Accuracy ({best_val_accuracy * 100:.2f}%)"
)

plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Plot training and validation loss

plt.figure(figsize=(10, 6))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

# Mark the best validation loss
best_loss_epoch = np.argmin(
    history.history["val_loss"]
)

best_val_loss = history.history["val_loss"][best_loss_epoch]

plt.scatter(
    best_loss_epoch,
    best_val_loss,
    s=80,
    label=f"Best Validation Loss ({best_val_loss:.4f})"
)

plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 21. Model Evaluation on the Test Dataset

The trained CNN is evaluated on the independent test dataset containing 3,631 images.

The test dataset was not used during model training or validation. Therefore, the test accuracy and loss provide an unbiased estimate of the model's performance on unseen images.

The following metrics are recorded:

- Test Loss
- Test Accuracy

In [ ]:
# Evaluate the trained CNN on the independent test dataset

test_loss, test_accuracy = model.evaluate(
    test_dataset,
    verbose=1
)

print("\nFinal Test Results")
print("-" * 40)

print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")

## 22. Generating Test Predictions

Predictions are generated for every image in the independent test dataset.

For each image, the CNN produces a probability distribution across the 10 tomato disease classes. The class with the highest probability is selected as the predicted class.

The following arrays are generated:

- `y_true`: Actual class labels
- `y_pred`: Predicted class labels
- `y_probability`: Confidence of the predicted class

In [ ]:
# Generate predictions for the complete test dataset

y_true = []
y_pred = []
y_probability = []

for batch_images, batch_labels in test_dataset:

    # Generate probability predictions
    predictions = model.predict(
        batch_images,
        verbose=0
    )

    # Select class with highest probability
    predicted_labels = np.argmax(
        predictions,
        axis=1
    )

    # Store actual labels
    y_true.extend(
        batch_labels.numpy()
    )

    # Store predicted labels
    y_pred.extend(
        predicted_labels
    )

    # Store prediction confidence
    y_probability.extend(
        np.max(predictions, axis=1)
    )


# Convert lists to NumPy arrays

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probability = np.array(y_probability)

print("Prediction generation completed.")
print("-" * 40)
print("Number of actual labels   :", len(y_true))
print("Number of predicted labels:", len(y_pred))
print("Number of confidence scores:", len(y_probability))

print("\nLabel range:")
print("Actual labels   :", y_true.min(), "to", y_true.max())
print("Predicted labels:", y_pred.min(), "to", y_pred.max())

## 23. Independent Test Accuracy Verification

The test accuracy is independently calculated using the actual labels and predicted labels.

This provides a second verification of the accuracy reported by `model.evaluate()`.

In [ ]:
# Calculate test accuracy independently

calculated_accuracy = accuracy_score(
    y_true,
    y_pred
)

print("Independent Accuracy Verification")
print("-" * 40)

print(
    f"Calculated Test Accuracy: "
    f"{calculated_accuracy * 100:.2f}%"
)

print(
    f"Model Evaluation Accuracy: "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"\nDifference: "
    f"{abs(calculated_accuracy - test_accuracy):.6f}"
)

## 24. Confusion Matrix

A confusion matrix is used to analyze the classification performance of the CNN across all 10 tomato disease categories.

The rows represent the actual classes, while the columns represent the predicted classes.

- Values along the diagonal represent correctly classified images.
- Off-diagonal values represent misclassified images.

The confusion matrix helps identify which disease categories are easily recognized and which categories are more frequently confused with one another.

In [ ]:
# Generate the confusion matrix

cm = confusion_matrix(
    y_true,
    y_pred
)

print("Confusion Matrix Shape:", cm.shape)
print("\nConfusion Matrix:")
print(cm)

# Visualize the confusion matrix

plt.figure(figsize=(13, 10))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=readable_class_names,
    yticklabels=readable_class_names
)

plt.title(
    "Confusion Matrix - Tomato Disease Classification",
    fontsize=15
)

plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.xticks(
    rotation=60,
    ha="right"
)

plt.yticks(
    rotation=0
)

plt.tight_layout()
plt.show()

## 25. Classification Report

The classification report provides detailed performance metrics for each tomato disease class.

The following metrics are calculated:

- **Precision:** Measures how many of the images predicted as a particular class actually belong to that class.
- **Recall:** Measures how many images belonging to a particular class are correctly identified.
- **F1-score:** Provides a combined measure of precision and recall.
- **Support:** Indicates the number of test images belonging to each class.

These metrics provide a more detailed evaluation than overall accuracy, particularly because the dataset is imbalanced.

In [ ]:
# Generate the classification report

classification_report_text = classification_report(
    y_true,
    y_pred,
    target_names=readable_class_names,
    digits=4
)

print("Classification Report")
print("=" * 80)
print(classification_report_text)

## 26. Detailed Classification Report

The classification report is converted into a Pandas DataFrame for easier interpretation and comparison of the performance across all tomato disease classes.

In [ ]:
# Convert the classification report into a dictionary

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=readable_class_names,
    output_dict=True
)

# Convert the dictionary into a DataFrame

report_df = pd.DataFrame(
    report_dict
).transpose()

# Display the report

display(
    report_df.round(4)
)

## 27. Sample Predictions

A selection of images from the independent test dataset is displayed with their actual class, predicted class, and prediction confidence.

This provides a visual demonstration of how the trained CNN performs on previously unseen tomato leaf images.

The prediction confidence represents the highest Softmax probability produced by the model for the predicted class.

In [ ]:
# Display sample predictions from the independent test dataset

# Select up to 12 random test images
num_samples = min(12, len(test_df))

sample_indices = np.random.choice(
    len(test_df),
    size=num_samples,
    replace=False
)

# Create visualization
fig, axes = plt.subplots(
    3,
    4,
    figsize=(16, 12)
)

axes = axes.flatten()

for plot_index, data_index in enumerate(sample_indices):

    # Get image path
    image_path = test_df.iloc[data_index]["filepath"]

    # Get actual label
    actual_label = test_df.iloc[data_index]["label"]

    # Get predicted label and confidence
    predicted_label = y_pred[data_index]
    confidence = y_probability[data_index] * 100

    # Load image
    image = plt.imread(image_path)

    # Display image
    axes[plot_index].imshow(image)

    # Determine whether prediction is correct
    prediction_status = (
        "Correct"
        if actual_label == predicted_label
        else "Incorrect"
    )

    axes[plot_index].set_title(
        f"Actual: {readable_class_names[actual_label]}\n"
        f"Predicted: {readable_class_names[predicted_label]}\n"
        f"Confidence: {confidence:.2f}%\n"
        f"{prediction_status}",
        fontsize=9
    )

    axes[plot_index].axis("off")


# Hide unused subplots
for i in range(num_samples, len(axes)):
    axes[i].axis("off")


plt.suptitle(
    "CNN Sample Predictions on Test Images",
    fontsize=16
)

plt.tight_layout()
plt.show()

## 28. Prediction Confidence Analysis

The confidence score represents the probability assigned by the CNN to its predicted class.

The average confidence across the complete test dataset is calculated. The number and percentage of correct and incorrect predictions are also reported.

This provides additional insight into how confidently the CNN makes its predictions.

In [ ]:
# Calculate prediction confidence statistics

correct_predictions = np.sum(
    y_true == y_pred
)

incorrect_predictions = np.sum(
    y_true != y_pred
)

total_predictions = len(y_true)

average_confidence = np.mean(
    y_probability
)

correct_confidences = y_probability[
    y_true == y_pred
]

incorrect_confidences = y_probability[
    y_true != y_pred
]

print("Prediction Confidence Analysis")
print("=" * 45)

print(f"Total predictions       : {total_predictions:,}")
print(f"Correct predictions     : {correct_predictions:,}")
print(f"Incorrect predictions   : {incorrect_predictions:,}")

print(
    f"Correct prediction rate : "
    f"{correct_predictions / total_predictions * 100:.2f}%"
)

print(
    f"Average confidence      : "
    f"{average_confidence * 100:.2f}%"
)

print(
    f"Average confidence "
    f"(correct predictions)  : "
    f"{correct_confidences.mean() * 100:.2f}%"
)

if len(incorrect_confidences) > 0:
    print(
        f"Average confidence "
        f"(incorrect predictions): "
        f"{incorrect_confidences.mean() * 100:.2f}%"
    )

## 29. Per-Class Error Analysis

The number of incorrect predictions is calculated for each tomato disease class.

This analysis helps identify the classes that are most difficult for the CNN to classify correctly and provides insight into the limitations of the model.

In [ ]:
# Calculate per-class error statistics

per_class_results = []

for class_index, class_name in enumerate(readable_class_names):

    # Find all test samples belonging to this class
    class_mask = y_true == class_index

    total_class_samples = np.sum(class_mask)

    correct_class_predictions = np.sum(
        (y_true == class_index) &
        (y_pred == class_index)
    )

    incorrect_class_predictions = (
        total_class_samples -
        correct_class_predictions
    )

    class_accuracy = (
        correct_class_predictions /
        total_class_samples
        if total_class_samples > 0
        else 0
    )

    per_class_results.append({
        "Class": class_name,
        "Total Samples": total_class_samples,
        "Correct": correct_class_predictions,
        "Incorrect": incorrect_class_predictions,
        "Accuracy": class_accuracy
    })


# Create DataFrame
per_class_error_df = pd.DataFrame(
    per_class_results
)

# Convert accuracy to percentage
per_class_error_df["Accuracy"] = (
    per_class_error_df["Accuracy"] * 100
)

# Sort by number of errors
per_class_error_df = per_class_error_df.sort_values(
    by="Incorrect",
    ascending=False
).reset_index(drop=True)

display(
    per_class_error_df.round(2)
)

## 30. Per-Class Accuracy

The accuracy of the CNN for each tomato disease class is visualized using a bar chart.

This comparison highlights the classes that the model recognizes most accurately and the classes where further improvement may be possible.

In [ ]:
# Sort classes alphabetically for visualization
accuracy_plot_df = per_class_error_df.sort_values(
    by="Accuracy",
    ascending=True
)

plt.figure(figsize=(12, 7))

plt.barh(
    accuracy_plot_df["Class"],
    accuracy_plot_df["Accuracy"]
)

plt.axvline(
    x=95,
    linestyle="--",
    label="95% Accuracy"
)

plt.title(
    "Per-Class Accuracy - Tomato Disease Classification",
    fontsize=15
)

plt.xlabel("Accuracy (%)")
plt.ylabel("Disease Class")

plt.xlim(0, 105)

plt.legend()

plt.tight_layout()
plt.show()

## 31. Precision, Recall, and F1-Score Comparison

Precision, recall, and F1-score are compared across all 10 tomato disease classes.

This visualization provides a detailed view of the model's classification performance beyond overall accuracy.

In [ ]:
# Extract class-wise metrics from the classification report

metrics_df = report_df.iloc[:NUM_CLASSES].copy()

# Create x-axis positions
x = np.arange(NUM_CLASSES)
width = 0.25

plt.figure(figsize=(15, 7))

plt.bar(
    x - width,
    metrics_df["precision"] * 100,
    width,
    label="Precision"
)

plt.bar(
    x,
    metrics_df["recall"] * 100,
    width,
    label="Recall"
)

plt.bar(
    x + width,
    metrics_df["f1-score"] * 100,
    width,
    label="F1-Score"
)

plt.title(
    "Precision, Recall and F1-Score by Disease Class",
    fontsize=15
)

plt.xlabel("Disease Class")
plt.ylabel("Score (%)")

plt.xticks(
    x,
    readable_class_names,
    rotation=60,
    ha="right"
)

plt.ylim(0, 105)

plt.legend()

plt.tight_layout()
plt.show()

## 32. Saving the Trained CNN Model

The trained CNN model is saved in Keras format so that it can be reused later for prediction or deployment without requiring retraining.

The model contains:

- CNN architecture
- Learned weights
- Optimizer configuration
- Training configuration

The saved model can later be loaded using TensorFlow/Keras.

In [ ]:
# Save the trained CNN model

MODEL_PATH = "tomato_disease_cnn.keras"

model.save(MODEL_PATH)

print("Model saved successfully.")
print("Model file:", MODEL_PATH)

# Verify that the saved model exists
if Path(MODEL_PATH).exists():
    model_size_mb = Path(MODEL_PATH).stat().st_size / (1024 * 1024)
    print(f"Model size: {model_size_mb:.2f} MB")
else:
    print("Warning: Model file was not found.")

## 33. Loading and Verifying the Saved Model

The saved Keras model is loaded back into memory and evaluated on the independent test dataset.

This verifies that the saved model preserves the learned architecture and weights and produces the same test performance as the original trained model.

In [ ]:
# Load the saved model

loaded_model = tf.keras.models.load_model(
    MODEL_PATH
)

print("Saved model loaded successfully.")

# Evaluate the loaded model
loaded_test_loss, loaded_test_accuracy = loaded_model.evaluate(
    test_dataset,
    verbose=1
)

print("\nLoaded Model Test Results")
print("-" * 40)

print(
    f"Test Loss     : {loaded_test_loss:.4f}"
)

print(
    f"Test Accuracy : {loaded_test_accuracy * 100:.2f}%"
)

print("\nVerification")
print("-" * 40)

print(
    f"Accuracy difference: "
    f"{abs(test_accuracy - loaded_test_accuracy):.6f}"
)

## 34. Final Results Summary

The CNN successfully classified 10 tomato leaf disease categories using the PlantVillage dataset.

The final model was evaluated on an independent test set containing 3,631 images.

### Final Performance

- Test Accuracy: **95.79%**
- Test Loss: **0.1362**
- Macro Precision: **94.00%**
- Macro Recall: **95.77%**
- Macro F1-Score: **94.76%**
- Weighted F1-Score: **95.82%**
- Correct Predictions: **3,478**
- Incorrect Predictions: **153**
- Average Prediction Confidence: **96.24%**

The best validation performance was obtained at **Epoch 22**, with a validation accuracy of **94.95%** and validation loss of **0.1839**.

The trained model was saved as `tomato_disease_cnn.keras` and successfully loaded again with no difference in test accuracy.

In [ ]:
# Create a consolidated final results summary

best_epoch_index = np.argmin(
    history.history["val_loss"]
)

best_epoch_number = best_epoch_index + 1

best_val_accuracy = history.history["val_accuracy"][
    best_epoch_index
]

best_val_loss = history.history["val_loss"][
    best_epoch_index
]

macro_precision = report_df.loc[
    "macro avg",
    "precision"
]

macro_recall = report_df.loc[
    "macro avg",
    "recall"
]

macro_f1 = report_df.loc[
    "macro avg",
    "f1-score"
]

weighted_f1 = report_df.loc[
    "weighted avg",
    "f1-score"
]


final_results = pd.DataFrame({
    "Metric": [
        "Number of Classes",
        "Training Images",
        "Validation Images",
        "Test Images",
        "Image Size",
        "Batch Size",
        "Maximum Epochs",
        "Best Epoch",
        "Best Validation Accuracy",
        "Best Validation Loss",
        "Test Accuracy",
        "Test Loss",
        "Macro Precision",
        "Macro Recall",
        "Macro F1-Score",
        "Weighted F1-Score",
        "Correct Predictions",
        "Incorrect Predictions",
        "Average Confidence"
    ],
    
    "Value": [
        NUM_CLASSES,
        len(train_df),
        len(validation_df),
        len(test_df),
        f"{IMG_HEIGHT} × {IMG_WIDTH} × {CHANNELS}",
        BATCH_SIZE,
        EPOCHS,
        best_epoch_number,
        f"{best_val_accuracy * 100:.2f}%",
        f"{best_val_loss:.4f}",
        f"{test_accuracy * 100:.2f}%",
        f"{test_loss:.4f}",
        f"{macro_precision * 100:.2f}%",
        f"{macro_recall * 100:.2f}%",
        f"{macro_f1 * 100:.2f}%",
        f"{weighted_f1 * 100:.2f}%",
        correct_predictions,
        incorrect_predictions,
        f"{average_confidence * 100:.2f}%"
    ]
})

display(final_results)

## 35. Conclusion

A Convolutional Neural Network (CNN) was successfully designed and implemented using TensorFlow/Keras for classifying tomato leaf diseases from the PlantVillage dataset.

The experiment used 10 tomato disease categories with 12,349 training images, 2,180 validation images, and 3,631 independent test images. Images were resized to 128 × 128 pixels and normalized before being provided to the CNN. Data augmentation and class weighting were used to improve generalization and reduce the effect of class imbalance.

The CNN achieved a **test accuracy of 95.79%** with a **test loss of 0.1362**. The model also achieved a **macro F1-score of 94.76%**, demonstrating strong performance across the different disease categories despite the imbalanced dataset.

The confusion matrix and classification report showed that most classes were classified accurately. Tomato mosaic virus achieved 100% recall, while the healthy class achieved a 99.37% F1-score. Some confusion remained between visually similar diseases such as Bacterial spot, Early blight, Late blight, and Spider mites.

The trained model was saved in Keras format and successfully loaded again, producing identical test accuracy. This confirms that the trained model can be reproduced and reused for future predictions.

Overall, the experiment demonstrates that CNNs can effectively learn visual features from plant leaf images and perform accurate multi-class disease classification.

In [ ]:
# Final experiment completion message

print("=" * 60)
print("CNN TOMATO DISEASE CLASSIFICATION - EXPERIMENT COMPLETE")
print("=" * 60)

print(f"\nNumber of classes       : {NUM_CLASSES}")
print(f"Training images         : {len(train_df):,}")
print(f"Validation images       : {len(validation_df):,}")
print(f"Test images             : {len(test_df):,}")

print(f"\nBest validation epoch   : {best_epoch_number}")
print(f"Best validation accuracy: {best_val_accuracy * 100:.2f}%")

print(f"\nFinal test accuracy     : {test_accuracy * 100:.2f}%")
print(f"Final test loss         : {test_loss:.4f}")
print(f"Macro F1-score          : {macro_f1 * 100:.2f}%")

print(f"\nCorrect predictions     : {correct_predictions:,}")
print(f"Incorrect predictions   : {incorrect_predictions:,}")

print(f"\nSaved model             : {MODEL_PATH}")

print("\n" + "=" * 60)
print("Experiment completed successfully!")
print("=" * 60)